# Banc d'essai `bench` — génération AP-HP

Notebook du nouveau monde `work_prompts/` (spec : `docs/spec_testrun_run_stage.md`, v3.6),
organisé selon le **cycle de vie d'un test** : initialisation → préparation →
run → bilan. Toutes les étapes qui touchent le disque sont **idempotentes** :
elles s'exécutent si nécessaire, sinon elles skippent avec un message d'état —
le notebook se ré-exécute intégralement sur un test déjà préparé.

Deux gestes à ne pas confondre :

- **MONTAGE** (section 2) : installer le **jeu de templates** du test dans
  `system/one_gen/` depuis la version précédente — c'est l'objet versionné,
  la chaîne des tests est la chaîne des versions des jeux.
- **SEEDING puis FIGEMENT** (section 3.1) : créer les **dossiers CRH** depuis
  les scénarios, puis résoudre le jeu par famille (`template.txt`) en
  `prompt_system_one_gen.txt` dans chaque dossier — c'est la production du run.

Rien ne se nettoie : pour repartir de zéro, on crée `tests/NN+1`. Le disque
fait foi ; **tout `work_prompts/` est versionné et partagé** (§9) — « figer »
un jeu = commiter son test.

La clé API vient **exclusivement** de l'environnement (`MISTRAL_API_KEY`),
jamais du notebook. Tout s'exécute **sans clé** jusqu'aux dry-runs inclus ;
seules les cellules « run réel » l'exigent.


## 1. Initialisation — déclarations pures

Bootstrap (`sys.path`, import de `bench`), paramétrage du test courant,
paramètres de sélection et de génération, client Mistral à la demande.
Aucune exécution lourde ici.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import sys

import polars as pl
from IPython.display import display


def _find_repo_root(start: Path) -> Path:
    """Racine du repo Stream : le dossier qui contient `bench/` et `core/`."""
    for candidate in (start, *start.parents):
        if (candidate / "bench").is_dir() and (candidate / "core").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Racine du repo Stream introuvable depuis {start} — lancer le "
        "notebook depuis work_prompts/ (ou un sous-dossier du repo)."
    )


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import bench
from bench import (
    BenchError,
    Pricing,
    copy_system_prompts,
    generate,
    load_reports,
    scenario_dirs,
    seed_user_prompts,
    summarize_costs,
    user_from_column,
    write_prompts,
)



def show_first_prompt(result, *, max_chars: int = 6000) -> None:
    """Affiche le premier prompt assemblé d'un `GenResult` dry-run.

    Contrôle à sec ET test de complétude : si `generate` a rendu la main,
    aucun fichier ne manquait dans aucun dossier scénario (§3.5).
    """
    if result.reports.height == 0:
        print("Aucun scénario retenu.")
        return
    row = result.reports.row(0, named=True)
    print(f"Scénario : {row['scenario']} — famille : {row['template']}")
    for label, key in (
        ("PROMPT SYSTÈME", "system_prompt"),
        ("PROMPT USER", "user_prompt"),
        ("PREFIX (assistant)", "prefix"),
    ):
        text = row[key] or ""
        print(f"\n{'=' * 28} {label} — {len(text)} caractère(s) {'=' * 28}")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"[... tronqué à {max_chars} caractères]")


print("Racine repo  :", REPO_ROOT)
print("bench        :", Path(bench.__file__).resolve().parent)


### Paramètres de sélection des scénarios

À garder **identiques d'un test à l'autre** : la comparabilité de la chaîne
repose sur la même graine (`SOURCE_FILTERS`, `RANDOM_SEED`, `TARGET_N`, ...).


In [ ]:
# Paramètres de la sélection — à renseigner, puis à garder IDENTIQUES d'un
# test à l'autre de la chaîne (comparabilité).
SOURCE_PROFILES_PATH = REPO_ROOT / "data/aphp/scenarios_bn_all_20260128.pq"  # à renseigner

SOURCE_FILTERS: list[dict] = [
    # ex. {"column": "diag2", "op": "endswith", "value": "8"},
]
CANDIDATE_POOL_SIZE = 200

SCENARIO_FILTERS: list[dict] = [
    # ex. {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]
TARGET_N = 10          # garde-fou : on est dans du test
RANDOM_SELECTION = True
RANDOM_SEED = 42


SCENARIO_DISPLAY_COLUMNS = [
    "generation_id",
    "template_name",
    "icd_primary_code",
    "ghm2",
    "admission_type",
]

### Paramètres de génération


In [ ]:
MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
PRICING = Pricing(
    batch_input_usd_per_million=0.25,
    batch_output_usd_per_million=0.75,
)

# Prefill de la première génération d'un test deux temps (annexe 2-gen, non
# utilisé — repris de l'ancien notebook).
FIRST_GEN_PREFIX = "Résumé clinique :"

# Injection du résumé intermédiaire dans le user prompt du second temps
# (annexe 2-gen, §4).
SUMMARY_HEADER = """### RÉSUMÉ CLINIQUE ISSU DE LA PREMIÈRE GÉNÉRATION
Le résumé ci-dessous est une aide intermédiaire.
Le scénario clinique, les codes, les fiches descriptives et les instructions
restent prioritaires en cas de divergence."""

SUMMARY_FOOTER = "### FIN DU RÉSUMÉ CLINIQUE INTERMÉDIAIRE"

# Test 3 — vérificateur : textes d'exemple, à adapter à la campagne.
VERIF_SYSTEM = """Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées."""

VERIF_USER = "Vérifie le compte rendu suivant et rends ton verdict."
VERIF_HEADER = "### COMPTE RENDU À VÉRIFIER"
VERIF_FOOTER = "### FIN DU COMPTE RENDU"

print("Modèle :", MODEL)
print("Tarifs batch ($ / 1M tokens) :", PRICING)

### Clé API — jamais en dur

La clé vient **exclusivement** de `os.environ["MISTRAL_API_KEY"]`, exportée **avant**
le lancement du kernel (`export MISTRAL_API_KEY=...` puis relancer Jupyter). Elle ne
doit jamais apparaître dans le notebook, ni dans aucun fichier versionné.

Le client n'est construit qu'au moment d'un **run réel** : les cellules dry-run
n'ont pas besoin de clé.

In [ ]:
def mistral_client():
    """Client Mistral construit à la demande — uniquement pour les runs réels."""
    try:
        api_key = os.environ["MISTRAL_API_KEY"]
    except KeyError:
        raise RuntimeError(
            "Variable d'environnement MISTRAL_API_KEY absente.\n"
            "Exporter la clé AVANT de lancer le kernel :\n"
            "    export MISTRAL_API_KEY=...    # puis relancer jupyter\n"
            "La clé ne doit JAMAIS être écrite dans ce notebook ni dans un "
            "fichier versionné."
        ) from None
    if not api_key.strip():
        raise RuntimeError("MISTRAL_API_KEY est définie mais vide.")
    from core.clients import MistralClient

    return MistralClient(api_key=api_key)


print("MISTRAL_API_KEY présente :", "MISTRAL_API_KEY" in os.environ)

## 2. Préparation du test — MONTAGE du jeu (file system, idempotent)

**Le jeu du test s'édite LÀ : `tests/<TEST_NUM>/system/one_gen/` — un `.txt`
par famille clinique.** Il est monté par copie du jeu du **test précédent**
(la chaîne des tests est la chaîne des versions) ; pour un tout premier test,
amorçage depuis `work_modif_prompts/template_one_gen`. Un `regles_atih.yml`
présent dans le jeu est copié tel quel (hors périmètre). Aucun appel fictomed
dans cette section — file system uniquement.


### Paramétrage du test courant


In [ ]:
# --- Paramétrage des chemins : SEUL endroit où un test est désigné ---
TEST_NUM = "02"    # le test courant
PREV_TEST = "01"   # test précédent de la chaîne (None pour un tout premier test)

WORK_DIR = REPO_ROOT / "work_prompts"
TESTS_DIR = WORK_DIR / "tests"  # versionné et partagé (§9)
TD = TESTS_DIR / TEST_NUM

print("Test courant :", TD, "(existe)" if TD.is_dir() else "(à créer)")
print(
    "Jeu amont    :",
    TESTS_DIR / PREV_TEST / "system" / "one_gen"
    if PREV_TEST
    else REPO_ROOT / "work_modif_prompts" / "template_one_gen",
)

In [ ]:
# État du test courant
_sys_dir = TD / "system" / "one_gen"
_scen = scenario_dirs(TD) if TD.is_dir() else []
print("Test               :", TD, "— présent" if TD.is_dir() else "— à créer")
print("Jeu system/one_gen :", "présent" if _sys_dir.is_dir() else "absent")
print("Dossiers scénario  :", len(_scen), _scen[:5], "…" if len(_scen) > 5 else "")
if _scen:
    _fige = (TD / _scen[0] / "prompt_system_one_gen.txt").is_file()
    print("Figement (1er dossier, prompt_system_one_gen.txt) :",
          "présent" if _fige else "absent")

In [ ]:
# Montage du jeu par la chaîne — skip si déjà monté
_sys_dir = TD / "system" / "one_gen"
if _sys_dir.is_dir():
    print("SKIP — jeu déjà monté :", _sys_dir)
else:
    src = (
        TESTS_DIR / PREV_TEST / "system" / "one_gen"
        if PREV_TEST
        else REPO_ROOT / "work_modif_prompts" / "template_one_gen"
    )
    # (historique : tests/01 utilisait la position "first" — sans importance,
    #  le notebook vise les tests futurs)
    TD.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, _sys_dir)
    print("MONTÉ :", _sys_dir, "<-", src)

## 3. Run du test

### 3.1 Écriture des prompts sur le disque — SEEDING puis FIGEMENT (idempotent)

Ordre impératif : **seeding** (création des dossiers CRH depuis les scénarios)
puis **figement** (résolution du jeu par famille dans chaque dossier). Chaque
étape s'exécute si sa production manque, sinon elle skippe avec son état.


#### a) Seeding — chaîne fictomed (voie nominale)

Les parquets de `data/aphp` contiennent des **profils sources** (PMSI), pas des
scénarios : c'est la chaîne fictomed qui produit les scénarios
(`generation_id`, `template_name`, `user_prompt`, `prefix`, ...). Elle est
entièrement **locale** — Mistral n'intervient qu'aux `generate()`, aucune clé
n'est nécessaire ici. Les fonctions sont importées de
`work_modif_prompts/aphp_generation_utils.py`, réutilisées **telles quelles** ;
leurs fichiers de travail vont sous `TD/.fictomed/` (dossier caché, hors
découverte). La frontière est nette : fictomed produit les scénarios, `bench`
commence après.

> **Prérequis** : fictomed installé en **éditable** depuis le clone
> `work_modif_prompts/_dependencies/fictomed_prompt_work` (branche
> `prompt-work`), comme le faisait le §1 de l'ancien notebook :
> `pip install -e work_modif_prompts/_dependencies/fictomed_prompt_work`.
> Le paquet PyPI `fictomed` 0.1.2 n'embarque pas `regles_atih.yml` et fait
> échouer la génération ; après un `uv sync` (qui réinstalle la version
> PyPI), refaire l'installation éditable.

fictomed lit ses chemins (profils d'entrée, sorties, référentiels) dans un
fichier de config `servers.yaml`, **généré ici pour le run** sous
`TD/.fictomed/`. Pendant la génération, le fichier de profils actif est
temporairement remplacé par les candidats sélectionnés puis **restauré**
(bloc `finally` déjà dans `generate_and_select_fictomed_scenarios`, backup
sous `.fictomed/_backups/`).


In [ ]:
# Seeding — skip si le test a déjà des dossiers scénario
_scen = scenario_dirs(TD) if TD.is_dir() else []
if _scen:
    print(f"SKIP — test déjà seedé : {len(_scen)} dossiers scénario.")
else:
    # utilitaires amont (ancien monde), importés à la demande
    import importlib.util

    _spec_utils = importlib.util.spec_from_file_location(
        "aphp_generation_utils",
        REPO_ROOT / "work_modif_prompts" / "aphp_generation_utils.py",
    )
    aphp_utils = importlib.util.module_from_spec(_spec_utils)
    _prev_dwb = sys.dont_write_bytecode
    sys.dont_write_bytecode = True  # pas de __pycache__ dans l'ancien monde
    try:
        _spec_utils.loader.exec_module(aphp_utils)
    finally:
        sys.dont_write_bytecode = _prev_dwb
    print("Utilitaires amont importés depuis :", _spec_utils.origin)

    # fichiers de travail fictomed sous TD/.fictomed/ (caché => hors découverte)
    FICTOMED_DIR = TD / ".fictomed"
    (FICTOMED_DIR / "_backups").mkdir(parents=True, exist_ok=True)

    _, _, _, candidate_source = aphp_utils.prepare_source_candidates(
        source_profiles_path=SOURCE_PROFILES_PATH,
        source_filters=SOURCE_FILTERS,
        candidate_pool_size=CANDIDATE_POOL_SIZE,
        random_selection=RANDOM_SELECTION,
        random_seed=RANDOM_SEED,
    )

    aphp_utils.write_fictomed_config(
        config_file=FICTOMED_DIR / "servers.yaml",
        project_root=REPO_ROOT,
        run_dir=FICTOMED_DIR,
    )

    _, _, selected_scenarios = aphp_utils.generate_and_select_fictomed_scenarios(
        candidate_source=candidate_source,
        config_file=FICTOMED_DIR / "servers.yaml",
        aphp_data_dir=REPO_ROOT / "data" / "aphp",
        paths={"backups": FICTOMED_DIR / "_backups"},
        scenario_filters=SCENARIO_FILTERS,
        target_n=TARGET_N,
        random_selection=RANDOM_SELECTION,
        random_seed=RANDOM_SEED,
        run_dir=FICTOMED_DIR,
    )
    display(selected_scenarios.select(SCENARIO_DISPLAY_COLUMNS))

    print(
        "Scénarios créés :",
        seed_user_prompts(TD, selected_scenarios, seed_path=SOURCE_PROFILES_PATH),
    )

#### b) Figement — résolution du jeu par scénario

`SYSTEM_PROMPT_FILE` est le nom du prompt système figé dans chaque dossier.
Pour **itérer** après édition du jeu : changer `SYSTEM_PROMPT_FILE`
(ex. `prompt_system_one_gen_v2.txt`) et `OUT_FILE` en conséquence (section
3.2, ex. `crh_v2.txt`) — les variantes coexistent, rien n'est écrasé.


In [ ]:
SYSTEM_PROMPT_FILE = "prompt_system_one_gen.txt"

# Figement — skip si le fichier figé est déjà présent dans tous les dossiers
_manquants = [n for n in scenario_dirs(TD) if not (TD / n / SYSTEM_PROMPT_FILE).is_file()]
if not _manquants:
    print(f"SKIP — {SYSTEM_PROMPT_FILE} déjà figé dans tous les dossiers.")
else:
    print("Scénarios servis :", copy_system_prompts(TD, "one_gen", dest=SYSTEM_PROMPT_FILE))

#### c) Prompts partagés du vérificateur (optionnel)

Posés une fois par `write_prompts` — skip s'ils sont déjà présents partout.


In [ ]:
for _fname, _text in (
    ("prompt_system_verif.txt", VERIF_SYSTEM),
    ("user_verification.txt", VERIF_USER),
):
    if all((TD / n / _fname).is_file() for n in scenario_dirs(TD)):
        print(f"SKIP — {_fname} déjà présent dans tous les dossiers.")
    else:
        print(f"{_fname} :", write_prompts(TD, _fname, _text))

### 3.2 Génération — contrôle à sec puis run réel

Le contrôle à sec ne fait aucun appel API et aucune écriture — pas besoin de
clé. C'est aussi le **test de complétude** : `generate` échoue (`BenchError`)
au moindre fichier manquant, aucun dossier n'est sauté en silence. Le run réel
écrase `OUT_FILE` à chaque re-run (geste normal) ; chaque run réel ajoute son
entrée au journal `usage.json`.


In [ ]:
OUT_FILE = "crh_generation.txt"

dry = generate(
    TD,
    system=SYSTEM_PROMPT_FILE,
    user="user_generation.txt",
    out=OUT_FILE,
    client=None,  # inutile en dry-run
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    dry_run=True,
)
show_first_prompt(dry)

In [ ]:
client = mistral_client()  # échoue ici, clairement, si MISTRAL_API_KEY absente

cr = generate(
    TD,
    system=SYSTEM_PROMPT_FILE,
    user="user_generation.txt",
    out=OUT_FILE,
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
)
print(cr.usage)

### 3.3 Vérificateur (optionnel)

Verdict nourri par les CR de la section 3.2 — depuis la mémoire (`cr.reports`)
ou le disque (`load_reports`). Reprise de session : kernel redémarré, rien en
mémoire, le disque fait foi — `strict=False` charge les scénarios déjà servis,
`only=` restreint le run à ceux-là (les autres dossiers restent intacts).


In [ ]:
try:
    ctx = load_reports(TD, OUT_FILE)
except BenchError:
    _names = scenario_dirs(TD)
    ctx = pl.DataFrame(
        {
            "scenario": _names,
            "report": ["[CR généré — placeholder de contrôle à sec]"] * len(_names),
        }
    )
    print(f"Pas de {OUT_FILE} sur disque : contexte placeholder.")

dry_verif = generate(
    TD,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=ctx,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    dry_run=True,
)
show_first_prompt(dry_verif)

In [ ]:
client = mistral_client()

cr_disque = load_reports(TD, OUT_FILE, strict=False)
if cr_disque.height == 0:
    raise RuntimeError(f"Aucun {OUT_FILE} sur disque : lancer d'abord le run réel (3.2).")

verdicts = generate(
    TD,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=cr_disque,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    only=cr_disque["scenario"].to_list(),
)
print(verdicts.usage)

## 4. Bilan — coûts et vérification mécanique

Journal **append-only** : chaque run réel ajoute une entrée, re-runs compris —
l'argent dépensé reste tracé même quand les sorties sont écrasées.
`committed_usd` = total engagé ; `current_usd` = coût de l'état courant.

Puis les contrôles mécaniques hors modèle (stdlib uniquement) : structure des
dossiers, schéma des CRH, formulations. Code retour non nul si au moins un
échec.


In [ ]:
if TESTS_DIR.is_dir():
    for _td in sorted(TESTS_DIR.iterdir()):
        if _td.is_dir() and not _td.name.startswith("."):
            print(f"=== {_td.name} — {_td} ===")
            display(summarize_costs(_td))
else:
    print("Aucun test encore créé sous", TESTS_DIR)

In [ ]:
!python {REPO_ROOT / "scripts" / "check_crh.py"} {TD}

In [ ]:
_reancre = REPO_ROOT / "scripts" / "reancre_crh.py"
if _reancre.is_file():
    !python {_reancre} {TD}
else:
    print("SKIP — script absent (pas encore commité) :", _reancre)

## Annexe A — deux générations (non utilisé)

Le workflow 2-gen (résumé puis CR : deux `generate`, le `reports` du premier
nourrissant le `context` du second, spec §4) n'est **pas utilisé** — conservé
pour le jour venu. Positions : `first_gen` et `second_gen` (amorçage depuis
`work_modif_prompts/template_first_gen` / `template_second_gen`, puis chaîne
comme le workflow principal). Cellules volontairement non exécutables — copier
dans des cellules code pour les activer.

```python
# Graine (test dédié) puis montage des DEUX positions
seed_user_prompts(TD, selected_scenarios, seed_path=SOURCE_PROFILES_PATH)
shutil.copytree(REPO_ROOT / "work_modif_prompts" / "template_first_gen",
                TD / "system" / "first_gen")
shutil.copytree(REPO_ROOT / "work_modif_prompts" / "template_second_gen",
                TD / "system" / "second_gen")

# [édition des jeux dans TD/system/first_gen/ et TD/system/second_gen/]
print("Figé first_gen  :", copy_system_prompts(TD, "first_gen"))
print("Figé second_gen :", copy_system_prompts(TD, "second_gen"))

# Premier temps — contrôle à sec (complétude) puis run réel
dry2a = generate(TD, system="prompt_system_first_gen.txt",
                 user="user_generation.txt", out="crh_resume.txt",
                 client=None, model=MODEL, max_tokens=MAX_TOKENS_SUMMARY,
                 pricing=PRICING, prefix_text=FIRST_GEN_PREFIX, dry_run=True)
show_first_prompt(dry2a)

res1 = generate(TD, system="prompt_system_first_gen.txt",
                user="user_generation.txt", out="crh_resume.txt",
                client=mistral_client(), model=MODEL,
                max_tokens=MAX_TOKENS_SUMMARY, pricing=PRICING,
                prefix_text=FIRST_GEN_PREFIX)

# Second temps — le résumé nourrit le contexte (à sec : placeholder possible,
# même geste que la reprise du §8)
cr2 = generate(TD, system="prompt_system_second_gen.txt",
               user="user_generation.txt", out="crh_final.txt",
               client=mistral_client(), model=MODEL,
               max_tokens=MAX_TOKENS_CR, pricing=PRICING,
               prefix_file="prefix.txt",  # prefill d'origine de la graine
               context=res1.reports,      # ou load_reports(TD, "crh_resume.txt")
               context_header=SUMMARY_HEADER, context_footer=SUMMARY_FOOTER)
print(cr2.usage)
```


## Annexe B — `prompt_local.py` — logique de user prompt locale au test (§3.7)

Pour tester une **construction** de user prompt différente sans toucher au
package : un `prompt_local.py` à la racine du test (c'est un fichier : la
découverte l'ignore), chargé ici et passé en `user_fn=` à `seed_user_prompts`
d'un **nouveau** test — changer `TEST_NUM`/`PREV_TEST` en tête de notebook,
puis utiliser la graine ci-dessous **à la place** de celle de la section 3 ;
la suite (montage, figement, dry-run) est le workflow principal, inchangé.
Hiérarchie des leviers : (1) éditer les `.txt` du test ; (2) `prompt_local.py` ;
(3) monkeypatch fictomed (fragile, à noter dans `test.json["notes"]`) ;
(4) modifier le clone fictomed éditable.


In [ ]:
TD.mkdir(parents=True, exist_ok=True)

_prompt_local_path = TD / "prompt_local.py"
if not _prompt_local_path.exists():  # ne jamais écraser une version éditée
    _prompt_local_path.write_text(
        '''"""User prompt local au test (spec §3.7) — exemple.

Point de départ possible : inspect.getsource sur la fonction fictomed
correspondante, copiée puis modifiée.
"""


def build_user(row: dict) -> str:
    """User prompt fictomed + rappel explicite du DP et du GHM."""
    return (
        str(row["user_prompt"]).rstrip()
        + "\\n\\nRappel codage : DP "
        + str(row.get("icd_primary_code"))
        + " — GHM "
        + str(row.get("ghm2"))
        + "\\n"
    )
''',
        encoding="utf-8",
    )
    print("Écrit :", _prompt_local_path)

import importlib.util

_spec_local = importlib.util.spec_from_file_location(
    f"prompt_local_{TEST_NUM}", _prompt_local_path
)
prompt_local = importlib.util.module_from_spec(_spec_local)
_prev_dwb = sys.dont_write_bytecode
sys.dont_write_bytecode = True  # pas de __pycache__ dans le dossier de test
try:
    _spec_local.loader.exec_module(prompt_local)
finally:
    sys.dont_write_bytecode = _prev_dwb

print("Chargé :", prompt_local.build_user.__doc__)

In [ ]:
# Graine avec user_fn — À LA PLACE de la graine de la section 3.1, sur un test
# neuf. Refus normal (BenchError) si le test courant est déjà seedé.
if "selected_scenarios" not in globals():
    print("SKIP — pas de scénarios en mémoire (la chaîne fictomed n'a pas "
          "tourné : test déjà seedé).")
else:
    try:
        print(
            "Scénarios créés :",
            seed_user_prompts(
                TD,
                selected_scenarios,
                user_fn=prompt_local.build_user,
                seed_path=SOURCE_PROFILES_PATH,
            ),
        )
    except BenchError as exc:
        print("Graine refusée (test déjà seedé — normal en re-run) :", exc)

## Annexe C — itérer par copie d'un dossier scénario

Un dossier scénario est autonome : sa copie emporte **tous** ses prompts (et ses
sorties éventuelles). Nom libre ; le prochain `generate` sur le test l'inclut
dans la découverte — `only=` permet de ne relancer que la copie.


In [ ]:
_names = scenario_dirs(TD)
_src = TD / _names[0]           # n'importe quel scénario servi
_dst = TD / f"{_src.name}_bis"  # nom libre
if _dst.exists():
    print("Copie déjà présente :", _dst)
else:
    shutil.copytree(_src, _dst)
    print("Copié :", _src.name, "->", _dst.name)
print("Découverte :", scenario_dirs(TD))

In [ ]:
# contrôle à sec — vérifie le prompt assemblé de la copie
dry_copie = generate(
    TD,
    system="prompt_system_one_gen.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=None,  # inutile en dry-run
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    only=[_dst.name],
    dry_run=True,
)
print(dry_copie.reports["system_prompt"][0][-1500:])

In [ ]:
# run réel, une seule requête — les autres dossiers restent intacts
cr_copie = generate(
    TD,
    system="prompt_system_one_gen.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=mistral_client(),  # la clé vient de l'environnement
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    only=[_dst.name],
)
print(cr_copie.usage)